0. Import Library & Tentukan File

In [1]:
import pandas as pd
import numpy as np
import re
file_path = "dataset/data-mentah-budgeting.xlsx"

1. Baca File Excel Setiap Sheet

In [2]:
# pilih hanya sheet data budget
sheet_budget = [
    "Kantor Pusat Silog",
    "LOGISTIK", "BUBI", "IKN",
    "VUBA", "VUDS", "VULS"]

all_sheets = pd.read_excel(file_path, sheet_name=sheet_budget, header=None)
print(all_sheets.keys())

dict_keys(['Kantor Pusat Silog', 'LOGISTIK', 'BUBI', 'IKN', 'VUBA', 'VUDS', 'VULS'])


In [3]:
all_sheets["Kantor Pusat Silog"].head()

,0,1,2,3,4,5,6,7,8
0,NaN,NaN,Funds Center/Commitment Item,Consumable Budget,Consumed Budget,Available Amount,Current Budget,Commitment/Actuals,NaN
1,NaN,NaN,Funds Ctr/Commtmt It,31741970345,25421474847,6320495498,60259175665,61231629313,SISA
2,001,GENERAL,001 GENERAL,233684600,231305770,2378830,1181453386,-11583363767,947768786
3,001,GENERAL,4121902006 PEND SEWA TRUCK,0,0,"0,0000",0,-84200000,0
4,001,GENERAL,4121909900 JASA LAINNYA,0,0,"0,0000",0,15000000,0


2. Buang 2 Baris Awal (Header SAP)

In [4]:
data_sheets = {}

for sheet_name, df in all_sheets.items():
    # ambil mulai baris data
    df = df.iloc[2:].copy()

    # reset index
    df.reset_index(drop=True, inplace=True)
    data_sheets[sheet_name] = df

In [5]:
data_sheets["Kantor Pusat Silog"].head()

,0,1,2,3,4,5,6,7,8
0,001,GENERAL,001 GENERAL,233684600,231305770,2378830,1181453386,-11583363767,947768786
1,001,GENERAL,4121902006 PEND SEWA TRUCK,0,0,"0,0000",0,-84200000,0
2,001,GENERAL,4121909900 JASA LAINNYA,0,0,"0,0000",0,15000000,0
3,001,GENERAL,6214000000 Biaya BBM kendaraan dinas,48618900,48612552,6348,233421610,48612552,184802710
4,001,GENERAL,6311000100 GAJI,0,0,"0,0000",0,250683834,0


3. Beri Nama Kolom

In [6]:
column_names = [
    "biro_code",
    "biro_name",
    "row_detail",
    "consumable_budget",
    "consumed_budget",
    "available_amount",
    "current_budget",
    "commitment_actuals",
    "sisa"]

for sheet_name, df in data_sheets.items():
    df.columns = column_names
    # tambahkan asal sheet
    df["source_entity"] = sheet_name.upper()
    data_sheets[sheet_name] = df

In [7]:
data_sheets["Kantor Pusat Silog"].head()

,biro_code,biro_name,row_detail,consumable_budget,consumed_budget,available_amount,current_budget,commitment_actuals,sisa,source_entity
0,001,GENERAL,001 GENERAL,233684600,231305770,2378830,1181453386,-11583363767,947768786,KANTOR PUSAT SILOG
1,001,GENERAL,4121902006 PEND SEWA TRUCK,0,0,"0,0000",0,-84200000,0,KANTOR PUSAT SILOG
2,001,GENERAL,4121909900 JASA LAINNYA,0,0,"0,0000",0,15000000,0,KANTOR PUSAT SILOG
3,001,GENERAL,6214000000 Biaya BBM kendaraan dinas,48618900,48612552,6348,233421610,48612552,184802710,KANTOR PUSAT SILOG
4,001,GENERAL,6311000100 GAJI,0,0,"0,0000",0,250683834,0,KANTOR PUSAT SILOG


4. Rapikan Data Code Biro

In [8]:
def clean_code_biro(value):
    if pd.isna(value):
        return None

    value = str(value).strip()
    
    # kasus hasil pandas: 1.0
    if value.endswith(".0"):
        value = value[:-2]

    # jika murni angka dan maksimal 3 digit
    # contoh 1 → 001, 10 → 010, 510 → 510
    if value.isdigit() and len(value) <= 3:
        value = value.zfill(3)

    return value

In [9]:
for sheet_name, df in data_sheets.items():
    df["biro_code"] = df["biro_code"].apply(clean_code_biro)
    data_sheets[sheet_name] = df

5. Splitting Detail Raw Menjadi Kode & Nama

In [10]:
def split_detail(value):
    if pd.isna(value):
        return pd.Series([None, None])

    value = str(value).strip()

    # split hanya pada whitespace pertama
    parts = value.split(maxsplit=1)
    code = parts[0].strip()

    if len(parts) > 1:
        name = parts[1].strip()
    else:
        name = None

    return pd.Series([code, name])

In [11]:
for sheet_name, df in data_sheets.items():
    df[["detail_code", "detail_name"]] = df["row_detail"].apply(split_detail)
    data_sheets[sheet_name] = df

In [12]:
data_sheets["Kantor Pusat Silog"][["row_detail", "detail_code", "detail_name"]].head()

,row_detail,detail_code,detail_name
0,001 GENERAL,001,GENERAL
1,4121902006 PEND SEWA TRUCK,4121902006,PEND SEWA TRUCK
2,4121909900 JASA LAINNYA,4121909900,JASA LAINNYA
3,6214000000 Biaya BBM kendaraan dinas,6214000000,Biaya BBM kendaraan dinas
4,6311000100 GAJI,6311000100,GAJI


6. Rapikan Angka Budget

In [13]:
def clean_number(value):

    if pd.isna(value):
        return 0

    # kalau sudah numeric
    if isinstance(value, (int, float, np.integer, np.floating)):
        return float(value)

    value = str(value).strip()

    if value == "":
        return 0

    # format indonesia:
    # 22.640.048,00
    if "," in value:
        value = value.replace(".", "")
        value = value.replace(",", ".")

    try:
        return float(value)

    except ValueError:
        return 0

In [14]:
budget_columns = [
    "consumable_budget",
    "consumed_budget",
    "available_amount",
    "current_budget",
    "commitment_actuals",
    "sisa"]

In [15]:
for sheet_name, df in data_sheets.items():
    for col in budget_columns:
        df[col] = df[col].apply(clean_number)
    data_sheets[sheet_name] = df

In [16]:
data_sheets["Kantor Pusat Silog"][["detail_code", "consumable_budget", "available_amount"]].head()

,detail_code,consumable_budget,available_amount
0,001,233684600.0,2378830.0
1,4121902006,0.0,0.0
2,4121909900,0.0,0.0
3,6214000000,48618900.0,6348.0
4,6311000100,0.0,0.0


7. Klasifikasikan Setiap Raw Detail

In [17]:
def classify_row(row):

    detail_code = row["detail_code"]
    parent_code = row["biro_code"]

    # row kosong
    if pd.isna(detail_code):
        return "Empty"

    detail_code = str(detail_code).strip()
    parent_code = str(parent_code).strip()

    # parent / summary
    if detail_code == parent_code:
        return "Parent"

    # funds center
    if detail_code.upper().startswith("S"):
        return "Funds_Center"

    # selain itu commitment item
    return "Commitment_Item"

In [18]:
for sheet_name, df in data_sheets.items():

    df["row_type"] = df.apply(
        classify_row,
        axis=1
    )

    data_sheets[sheet_name] = df

In [19]:
data_sheets["BUBI"][["biro_code", "biro_name", "detail_code", "detail_name", "row_type"]]

,biro_code,biro_name,detail_code,detail_name,row_type
0,001,GENERAL,001,GENERAL,Parent
1,001,GENERAL,1162000300,BDP NON STOCK,Commitment_Item
2,310,BIRO MANAJEMEN RANTAI PASOK,310,BIRO MANAJEMEN RANTAI PASOK,Parent
3,310,BIRO MANAJEMEN RANTAI PASOK,1162000300,BDP NON STOCK,Commitment_Item
4,310,BIRO MANAJEMEN RANTAI PASOK,6422000000,Biaya Konsultan,Commitment_Item
...,...,...,...,...,...
127,410,BIRO AKUNTANSI,6541000400,PEMELIHARAAN TRUK,Commitment_Item
128,410,BIRO AKUNTANSI,S004TL0124,TRUCK SILOG NOPIN TL124,Funds_Center
129,410,BIRO AKUNTANSI,6541000400,PEMELIHARAAN TRUK,Commitment_Item
130,410,BIRO AKUNTANSI,S004W00B06,TRUCK SILOG NOPIN WB06,Funds_Center


8. Buat Current Funds Center

In [20]:
# setiap bertemu row funds center, maka disimpan sebagai konteks untuk ci setelahnya
processed_sheets = {}

for sheet_name, df in data_sheets.items():
    df = df.copy()

    current_parent = None
    current_fc = None
    current_fc_name = None

    mapped_fc = []
    mapped_fc_name = []

    for _, row in df.iterrows():
        parent_code = row["biro_code"]
        row_type = row["row_type"]

        # kalau parent berubah, reset FC
        if parent_code != current_parent:
            current_parent = parent_code
            current_fc = None
            current_fc_name = None

        # parent summary
        if row_type == "Parent":
            current_fc = None
            current_fc_name = None

        # funds center
        elif row_type == "Funds_Center":
            current_fc = row["detail_code"]
            current_fc_name = row["detail_name"]

        mapped_fc.append(current_fc)
        mapped_fc_name.append(current_fc_name)

    df["current_funds_center"] = mapped_fc
    df["current_funds_center_name"] = mapped_fc_name

    processed_sheets[sheet_name] = df

9. Pengecekan Hasil Current Funds Center

In [21]:
# pengecekan
for sheet_name, df in processed_sheets.items():

    print(f"\n===== {sheet_name.upper()} =====")

    display(df[[
                "biro_code",
                "detail_code",
                "detail_name",
                "row_type",
                "current_funds_center"]].head(50))


===== KANTOR PUSAT SILOG =====


,biro_code,detail_code,detail_name,row_type,current_funds_center
0,001,001,GENERAL,Parent,None
1,001,4121902006,PEND SEWA TRUCK,Commitment_Item,None
2,001,4121909900,JASA LAINNYA,Commitment_Item,None
3,001,6214000000,Biaya BBM kendaraan dinas,Commitment_Item,None
4,001,6311000100,GAJI,Commitment_Item,None
5,001,6311000200,HONORARIUM,Commitment_Item,None
6,001,6313000300,TUNJANGAN MAKAN,Commitment_Item,None
7,001,6313001100,TUNJANGAN KEAGAMAAN,Commitment_Item,None
8,001,6313001901,TUNJANGAN LAINNYA,Commitment_Item,None
9,001,6313001902,TUNJANGAN PENGAWASAN,Commitment_Item,None



===== LOGISTIK =====


,biro_code,detail_code,detail_name,row_type,current_funds_center
0,220,220,BIRO PENGADAAN BARANG & JASA,Parent,None
1,220,6710000000,Biaya Perjalanan Dinas,Commitment_Item,None
2,220,6771000100,SEWA TANAH,Commitment_Item,None
3,220,6771000200,SEWA BANGUNAN & PRASARANA,Commitment_Item,None
4,220,68120004001,ONGKOS PEMUATAN KE TRUK,Commitment_Item,None
5,220,7154000100,BEBAN KLAIM,Commitment_Item,None
6,221,221,SEKSI PENGADAAN LOGISTIK & FABRIKASI,Parent,None
7,221,67220047001,KEPERLUAN CLOSING,Commitment_Item,None
8,250,250,"DEPARTMENT OF SAFETY, HEALTH & ENVIRONME",Parent,None
9,250,6214000000,Biaya BBM kendaraan dinas,Commitment_Item,None



===== BUBI =====


...

,biro_code,detail_code,detail_name,row_type,current_funds_center
0,510,510,BAGIAN OPERASIONAL,Parent,None
1,510,4121902008,PEND BONGKAR MUAT,Commitment_Item,None
2,510,4121902015,PENDAPATAN TUNDA PANDU,Commitment_Item,None
3,510,4121909900,JASA LAINNYA,Commitment_Item,None
4,510,5911000303,BIAYA ONGKOS PEMUATAN,Commitment_Item,None
5,510,5911000307,BIAYA SEWA ALAT BERAT,Commitment_Item,None
6,510,5911000310,BIAYA KULI PELABUHAN,Commitment_Item,None
7,510,5911000312,HPP LAINNYA,Commitment_Item,None
8,510,5911000349,BIAYA OPS TUNDA PANDU LAINNYA,Commitment_Item,None
9,510,5911000352,BIAYA PENGELOLAAN DERMAGA,Commitment_Item,None



===== VUDS =====


,biro_code,detail_code,detail_name,row_type,current_funds_center
0,340,340,BIRO MARKETING NON SEMEN,Parent,None
1,340,68120004001,ONGKOS PEMUATAN KE TRUK,Commitment_Item,None
2,410,410,BIRO AKUNTANSI,Parent,None
3,410,6784000100,BEBAN ADMINISTRASI BANK,Commitment_Item,None
4,610,610,FORWARDING AND NON REGULER SERVICE SECTI,Parent,None
5,610,1182000600,UANG MUKA OPERASIONAL KARYAWAN,Commitment_Item,None
6,610,4121902005,JASA ANGKUTAN TRUCK,Commitment_Item,None
7,610,4121902009,PEND CUSTOM CLEARENCE & FWD,Commitment_Item,None
8,610,4121902019,Pend. Pengelolaan Gudang,Commitment_Item,None
9,610,4121909900,JASA LAINNYA,Commitment_Item,None



===== VULS =====


,biro_code,detail_code,detail_name,row_type,current_funds_center
0,710,710,BIDANG USAHA PELAYARAN,Parent,None
1,710,1617009900,PEKERJAAN DLM.PELAKSANAAN-LAINNYA,Commitment_Item,None
2,710,4121902011,PEND PELAYARAN,Commitment_Item,None
3,710,4121902012,PEND KEAGENAN,Commitment_Item,None
4,710,4121902013,PEND KEBUTUHAN KAPAL,Commitment_Item,None
5,710,5911000344,BIAYA LAINNYA PELAYARAN,Commitment_Item,None
6,710,6214000000,Biaya BBM kendaraan dinas,Commitment_Item,None
7,710,62140001001,PEMAKAIAN SOLAR,Commitment_Item,None
8,710,6220000000,Biaya Listrik & PDAM,Commitment_Item,None
9,710,6311000100,GAJI,Commitment_Item,None


10. Ambil Raw Commitment Item

In [22]:
# Pisahkan sumber DETAIL untuk perhitungan budget, tanpa kehilangan PARENT.
# processed_sheets tetap menyimpan seluruh baris dan urutan asli.
cleaned_sheets = {}
for sheet_name, df in processed_sheets.items():
    cleaned_sheets[sheet_name] = df.loc[df["row_type"] == "Commitment_Item"].copy()


11. Menentukan Tipe Budget

In [23]:
#S004BL0139 + 6541000400 → FC_CI
# NULL + 6313001910      → CI_ONLY
for sheet_name, df in cleaned_sheets.items():
    df["budget_scope"] = np.where(
        df["current_funds_center"].notna(),"FC_CI", "CI_ONLY")
    cleaned_sheets[sheet_name] = df

12. Merapikan Nama Kolom Final

In [24]:
# Kolom yang sama dipakai baik untuk PARENT maupun DETAIL.
# Parent disimpan sebagai audit, bukan sebagai anggaran transaksi.
output_cols = [
    "source_entity", "biro_code", "biro_name", "row_type",
    "current_funds_center", "current_funds_center_name",
    "detail_code", "detail_name", "budget_scope",
    "consumable_budget", "consumed_budget", "available_amount",
    "current_budget", "commitment_actuals", "sisa"
]
rename_cols = {
    "current_funds_center": "funds_center_code",
    "current_funds_center_name": "funds_center_name",
    "detail_code": "commitment_item_code",
    "detail_name": "commitment_item_name"
}
final_sheets = {}
all_audit_sheets = {}
for sheet_name, df in cleaned_sheets.items():
    final_df = df[output_cols].copy().rename(columns=rename_cols)
    final_df["row_type"] = "DETAIL"
    final_sheets[sheet_name] = final_df

for sheet_name, df in processed_sheets.items():
    # Tetap berurutan seperti data sumber; keluarkan baris heading Funds Center
    # dan baris Empty karena bukan PARENT maupun detail Commitment Item.
    audit = df.loc[df["row_type"].isin(["Parent", "Commitment_Item"])].copy()
    audit["budget_scope"] = np.where(
        audit["row_type"].eq("Commitment_Item"),
        np.where(audit["current_funds_center"].notna(), "FC_CI", "CI_ONLY"),
        None
    )
    audit = audit[output_cols].rename(columns=rename_cols)
    audit["row_type"] = audit["row_type"].replace({
        "Parent": "PARENT", "Commitment_Item": "DETAIL"
    })
    all_audit_sheets[sheet_name] = audit


13. Gabungkan Seluruh Sheet

In [25]:
# clean_budget: PARENT + DETAIL; bukan untuk import langsung!
clean_budget = pd.concat(all_audit_sheets.values(), ignore_index=True)
# budget_detail: hanya DETAIL, dipakai untuk validasi dan import budget_master.
budget_detail = pd.concat(final_sheets.values(), ignore_index=True)
print('Seluruh baris PARENT + DETAIL:', len(clean_budget))
print('Baris DETAIL untuk database:', len(budget_detail))
print(clean_budget['row_type'].value_counts(dropna=False))


Seluruh baris PARENT + DETAIL: 7563
Baris DETAIL untuk database: 7504
row_type
DETAIL    7504
PARENT      59
Name: count, dtype: int64


In [26]:
display(clean_budget.head(20))

,source_entity,biro_code,biro_name,row_type,funds_center_code,funds_center_name,commitment_item_code,commitment_item_name,budget_scope,consumable_budget,consumed_budget,available_amount,current_budget,commitment_actuals,sisa
0,KANTOR PUSAT SILOG,001,GENERAL,PARENT,None,None,001,GENERAL,None,233684600.0,231305770.0,2378830.0,1.181453e+09,-1.158336e+10,947768786.0
1,KANTOR PUSAT SILOG,001,GENERAL,DETAIL,None,None,4121902006,PEND SEWA TRUCK,CI_ONLY,0.0,0.0,0.0,0.000000e+00,-8.420000e+07,0.0
2,KANTOR PUSAT SILOG,001,GENERAL,DETAIL,None,None,4121909900,JASA LAINNYA,CI_ONLY,0.0,0.0,0.0,0.000000e+00,1.500000e+07,0.0
3,KANTOR PUSAT SILOG,001,GENERAL,DETAIL,None,None,6214000000,Biaya BBM kendaraan dinas,CI_ONLY,48618900.0,48612552.0,6348.0,2.334216e+08,4.861255e+07,184802710.0
4,KANTOR PUSAT SILOG,001,GENERAL,DETAIL,None,None,6311000100,GAJI,CI_ONLY,0.0,0.0,0.0,0.000000e+00,2.506838e+08,0.0
5,KANTOR PUSAT SILOG,001,GENERAL,DETAIL,None,None,6311000200,HONORARIUM,CI_ONLY,0.0,0.0,0.0,0.000000e+00,3.345229e+09,0.0
6,KANTOR PUSAT SILOG,001,GENERAL,DETAIL,None,None,6313000300,TUNJANGAN MAKAN,CI_ONLY,0.0,0.0,0.0,0.000000e+00,1.386000e+07,0.0
7,KANTOR PUSAT SILOG,001,GENERAL,DETAIL,None,None,6313001100,TUNJANGAN KEAGAMAAN,CI_ONLY,0.0,0.0,0.0,0.000000e+00,1.599873e+08,0.0
8,KANTOR PUSAT SILOG,001,GENERAL,DETAIL,None,None,6313001901,TUNJANGAN LAINNYA,CI_ONLY,0.0,0.0,0.0,0.000000e+00,9.761387e+07,0.0
9,KANTOR PUSAT SILOG,001,GENERAL,DETAIL,None,None,6313001902,TUNJANGAN PENGAWASAN,CI_ONLY,0.0,0.0,0.0,0.000000e+00,4.366580e+07,0.0


14. Jumlah FC_CI & CI_ONLY

In [27]:
print('Detail berdasarkan budget_scope:')
print(budget_detail["budget_scope"].value_counts(dropna=False))
print('Parent/Detail:')
print(clean_budget["row_type"].value_counts(dropna=False))


Detail berdasarkan budget_scope:
budget_scope
FC_CI      6467
CI_ONLY    1037
Name: count, dtype: int64
Parent/Detail:
row_type
DETAIL    7504
PARENT      59
Name: count, dtype: int64


15. Cek Duplikat

In [28]:
fc_ci = budget_detail.loc[budget_detail["budget_scope"] == "FC_CI"].copy()
duplicate_fc_ci = fc_ci.duplicated(
    subset=["source_entity", "biro_code", "funds_center_code", "commitment_item_code"],
    keep=False
)
display(fc_ci.loc[duplicate_fc_ci])


,source_entity,biro_code,biro_name,row_type,funds_center_code,funds_center_name,commitment_item_code,commitment_item_name,budget_scope,consumable_budget,consumed_budget,available_amount,current_budget,commitment_actuals,sisa


In [29]:
ci_only = budget_detail.loc[budget_detail["budget_scope"] == "CI_ONLY"].copy()
duplicate_ci = ci_only.duplicated(
    subset=["source_entity", "biro_code", "commitment_item_code"],
    keep=False
)
display(ci_only.loc[duplicate_ci])


,source_entity,biro_code,biro_name,row_type,funds_center_code,funds_center_name,commitment_item_code,commitment_item_name,budget_scope,consumable_budget,consumed_budget,available_amount,current_budget,commitment_actuals,sisa


16. Validasi Perhitungan Budget

In [30]:
# Lakukan perhitungan hanya pada DETAIL, bukan baris subtotal parent.
budget_detail["calculated_available"] = (
    budget_detail["consumable_budget"] - budget_detail["consumed_budget"]
)
budget_detail["available_difference"] = (
    budget_detail["available_amount"] - budget_detail["calculated_available"]
)
# Sheet clean_budget untuk audit juga menyimpan perhitungan agar dapat ditinjau.
clean_budget["calculated_available"] = (
    clean_budget["consumable_budget"] - clean_budget["consumed_budget"]
)
clean_budget["available_difference"] = (
    clean_budget["available_amount"] - clean_budget["calculated_available"]
)


In [31]:
beda = budget_detail.loc[budget_detail["available_difference"].abs() > 1]
display(beda)


,source_entity,biro_code,biro_name,row_type,funds_center_code,funds_center_name,commitment_item_code,commitment_item_name,budget_scope,consumable_budget,consumed_budget,available_amount,current_budget,commitment_actuals,sisa,calculated_available,available_difference
113,KANTOR PUSAT SILOG,230,BIRO HUKUM & MANAJEMEN RISIKO,DETAIL,None,None,6710000000,Biaya Perjalanan Dinas,CI_ONLY,92571817.0,87641011.0,2230806.0,92602994.0,87674011.0,31177.0,4930806.0,-2700000.0


17. Export Hasil Cleaning

In [32]:
from pathlib import Path
# Jalankan dari folder notebooks (data berada di folder proyek yang sejajar).
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUTPUT_DIR = ROOT / "data" / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
output_file = OUTPUT_DIR / "clean-budget.xlsx"

# Validasi pemisahan supaya subtotal PARENT tidak ikut terhitung sebagai budget.
assert (budget_detail["row_type"] == "DETAIL").all()
assert len(budget_detail) == int((clean_budget["row_type"] == "DETAIL").sum())
assert not budget_detail["budget_scope"].isna().any()

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    clean_budget.to_excel(writer, sheet_name="clean_budget", index=False)
    budget_detail.to_excel(writer, sheet_name="budget_detail", index=False)
    fc_ci.to_excel(writer, sheet_name="FC_CI", index=False)
    ci_only.to_excel(writer, sheet_name="CI_ONLY", index=False)

print('Selesai:', output_file)
print('PARENT:', int((clean_budget['row_type'] == 'PARENT').sum()))
print('DETAIL:', len(budget_detail))


Selesai: /Users/rrezaputria/Documents/GitHub/13okt-input database/13-data preparing/data/processed/clean-budget.xlsx
PARENT: 59
DETAIL: 7504
